In [0]:
# ============================================================
# Configuration
# ============================================================

CATALOG = "worldbank_ai"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Source Bronze observation table
OBSERVATIONS_SOURCE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.indicator_observations_raw"
)

# Silver dimensions
COUNTRIES_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.countries"
)

INDICATOR_METADATA_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.indicator_metadata"
)

# Target Silver fact table
TARGET_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.indicator_observations"
)

# Requested observation period
START_YEAR = 2010
END_YEAR = 2026

print(f"Observations source: {OBSERVATIONS_SOURCE}")
print(f"Countries dimension: {COUNTRIES_TABLE}")
print(f"Indicator dimension: {INDICATOR_METADATA_TABLE}")
print(f"Target:              {TARGET_TABLE}")
print(f"Expected years:      {START_YEAR}-{END_YEAR}")

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# ============================================================
# Load Bronze observations and Silver dimensions
# ============================================================

bronze_observations_df = spark.table(
    OBSERVATIONS_SOURCE
)

countries_df = spark.table(
    COUNTRIES_TABLE
)

indicator_metadata_df = spark.table(
    INDICATOR_METADATA_TABLE
)

# Record counts
bronze_count = bronze_observations_df.count()
country_count = countries_df.count()
indicator_count = indicator_metadata_df.count()

print(f"Bronze observations: {bronze_count:,}")
print(f"Silver entities:     {country_count:,}")
print(f"Silver indicators:   {indicator_count:,}")

In [0]:
# ============================================================
# Inspect source and dimension schemas
# ============================================================

print("=" * 70)
print("BRONZE OBSERVATIONS")
print("=" * 70)

bronze_observations_df.printSchema()


print("\n" + "=" * 70)
print("SILVER COUNTRIES")
print("=" * 70)

countries_df.printSchema()


print("\n" + "=" * 70)
print("SILVER INDICATOR METADATA")
print("=" * 70)

indicator_metadata_df.printSchema()

In [0]:
# ============================================================
# Validate required observation keys
# ============================================================
#
# Every observation must have:
#   - entity_id
#   - indicator_id
#   - year_raw
#
# value is allowed to be NULL because missing World Bank
# observations are legitimate missing data.
# ============================================================

bronze_key_validation = (
    bronze_observations_df
    .select(

        F.sum(
            F.when(
                F.col("entity_id").isNull()
                | (F.length(F.trim(F.col("entity_id"))) == 0),
                1
            ).otherwise(0)
        ).alias("missing_entity_id"),

        F.sum(
            F.when(
                F.col("indicator_id").isNull()
                | (F.length(F.trim(F.col("indicator_id"))) == 0),
                1
            ).otherwise(0)
        ).alias("missing_indicator_id"),

        F.sum(
            F.when(
                F.col("year_raw").isNull()
                | (F.length(F.trim(F.col("year_raw"))) == 0),
                1
            ).otherwise(0)
        ).alias("missing_year")
    )
    .collect()[0]
)

missing_entity_ids = bronze_key_validation["missing_entity_id"]
missing_indicator_ids = bronze_key_validation["missing_indicator_id"]
missing_years = bronze_key_validation["missing_year"]

print(f"Missing entity IDs:    {missing_entity_ids}")
print(f"Missing indicator IDs: {missing_indicator_ids}")
print(f"Missing years:         {missing_years}")

if (
    missing_entity_ids > 0
    or missing_indicator_ids > 0
    or missing_years > 0
):
    raise RuntimeError(
        "Required Bronze observation keys are missing."
    )

print("Bronze observation key validation passed.")

In [0]:
# ============================================================
# Standardize Bronze observation fields
# ============================================================
#
# World Bank observation endpoint:
#
#   entity_id        -> ISO2-style/source identifier
#                       Example: IN
#
#   entity_iso3_code -> ISO3 when available
#                       Example: IND
#
# Some World Bank aggregates do not have ISO3 values,
# therefore entity_iso3_code is allowed to be NULL.
# ============================================================

observations_clean_df = (
    bronze_observations_df

    # Clean source entity identifier
    .withColumn(
        "entity_id_clean",
        F.upper(
            F.trim(F.col("entity_id"))
        )
    )

    # Clean ISO3 where available
    .withColumn(
        "entity_iso3_code_clean",
        F.when(
            F.col("entity_iso3_code").isNull()
            | (
                F.length(
                    F.trim(F.col("entity_iso3_code"))
                ) == 0
            ),
            None
        )
        .otherwise(
            F.upper(
                F.trim(F.col("entity_iso3_code"))
            )
        )
    )

    # Standardize indicator identifier
    .withColumn(
        "indicator_id_clean",
        F.upper(
            F.trim(F.col("indicator_id"))
        )
    )

    # Safely convert source year string to integer
    .withColumn(
        "year",
        F.expr(
            "try_cast(trim(year_raw) AS INT)"
        )
    )

    # Silver processing timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# Validate year conversion
# ============================================================
#
# try_cast prevents malformed values from crashing the
# transformation, but we explicitly check whether any
# conversion failed.
# ============================================================

invalid_year_conversion_df = (
    observations_clean_df
    .filter(
        F.col("year_raw").isNotNull()
        & (F.length(F.trim(F.col("year_raw"))) > 0)
        & F.col("year").isNull()
    )
)

invalid_year_conversion_count = (
    invalid_year_conversion_df.count()
)

print(
    f"Failed year conversions: "
    f"{invalid_year_conversion_count:,}"
)

if invalid_year_conversion_count > 0:

    display(
        invalid_year_conversion_df
        .select(
            "entity_id",
            "indicator_id",
            "year_raw"
        )
    )

    raise RuntimeError(
        "One or more years could not be converted to integer."
    )

print("Year conversion validation passed.")

In [0]:
# ============================================================
# Validate observation year range
# ============================================================

out_of_range_years_df = (
    observations_clean_df
    .filter(
        (F.col("year") < START_YEAR)
        | (F.col("year") > END_YEAR)
    )
)

out_of_range_count = (
    out_of_range_years_df.count()
)

print(
    f"Out-of-range observations: "
    f"{out_of_range_count:,}"
)

if out_of_range_count > 0:

    display(
        out_of_range_years_df
        .select(
            "entity_id_clean",
            "indicator_id_clean",
            "year"
        )
    )

    raise RuntimeError(
        f"Observations outside "
        f"{START_YEAR}-{END_YEAR} detected."
    )

print("Year-range validation passed.")

In [0]:
# ============================================================
# Validate source observation grain
# ============================================================
#
# Expected grain:
#
#   source entity + indicator + year
#
# There should be only one observation for each combination.
# ============================================================

duplicate_observations_df = (
    observations_clean_df

    .groupBy(
        "entity_id_clean",
        "indicator_id_clean",
        "year"
    )

    .count()

    .filter(
        F.col("count") > 1
    )
)

duplicate_observation_count = (
    duplicate_observations_df.count()
)

print(
    f"Duplicate entity-indicator-year keys: "
    f"{duplicate_observation_count:,}"
)

if duplicate_observation_count > 0:

    display(
        duplicate_observations_df
    )

    raise RuntimeError(
        "Duplicate source observation grain detected."
    )

print("Observation grain validation passed.")

In [0]:
# ============================================================
# Prepare Silver country/entity dimension
# ============================================================
#
# IMPORTANT SOURCE RELATIONSHIP:
#
# Observation endpoint:
#     India -> entity_id = IN
#
# Silver country dimension:
#     India -> entity_id = IND
#              iso2_code = IN
#
# Therefore:
#
#     observations.entity_id
#              =
#     countries.iso2_code
#
# After resolving the entity, the final fact table will use
# the canonical Silver entity_id (IND).
# ============================================================

country_lookup_df = (
    countries_df

    .select(

        # Canonical Silver identifier
        F.col("entity_id").alias(
            "canonical_entity_id"
        ),

        # Join key used by observation endpoint
        F.upper(
            F.trim(F.col("iso2_code"))
        ).alias(
            "country_iso2_code"
        ),

        # Canonical entity name
        F.col("entity_name").alias(
            "dimension_entity_name"
        ),

        # Entity classification
        "entity_type",

        # Geographic classifications
        "region_id",
        "region_name",

        "admin_region_id",
        "admin_region_name",

        # Income classification
        "income_level_id",
        "income_level_name",

        # Lending classification
        "lending_type_id",
        "lending_type_name"
    )
)

print(
    f"Country lookup records: "
    f"{country_lookup_df.count():,}"
)

In [0]:
# ============================================================
# Validate ISO2 join-key uniqueness
# ============================================================
#
# Before joining 63,600 fact records, prove that every
# non-null ISO2 code maps to at most one dimension entity.
#
# Otherwise a join could multiply observation rows.
# ============================================================

duplicate_iso2_df = (
    country_lookup_df

    .filter(
        F.col("country_iso2_code").isNotNull()
    )

    .groupBy(
        "country_iso2_code"
    )

    .count()

    .filter(
        F.col("count") > 1
    )
)

duplicate_iso2_count = (
    duplicate_iso2_df.count()
)

print(
    f"Duplicate ISO2 join keys: "
    f"{duplicate_iso2_count:,}"
)

if duplicate_iso2_count > 0:

    display(
        duplicate_iso2_df
    )

    raise RuntimeError(
        "Silver country ISO2 codes are not unique."
    )

print(
    "Country ISO2 join-key validation passed."
)

In [0]:
# ============================================================
# Prepare Silver indicator dimension
# ============================================================
#
# Select only metadata needed by the observation fact table.
# Aliases prevent duplicate column names during the join.
# ============================================================

indicator_lookup_df = (
    indicator_metadata_df

    .select(

        F.col("indicator_id").alias(
            "metadata_indicator_id"
        ),

        F.col("display_name").alias(
            "indicator_display_name"
        ),

        F.col("official_name").alias(
            "indicator_official_name"
        ),

        F.col("unit").alias(
            "metadata_unit"
        ),

        "indicator_category"
    )
)

In [0]:
# ============================================================
# Enrich observations with country/entity metadata
# ============================================================
#
# Correct join discovered through source validation:
#
#     observation entity_id = country iso2_code
#
# Example:
#
#     IN = IN
#
# LEFT JOIN is intentional.
#
# If a dimension record is missing, we want to detect it
# through validation rather than silently delete observations.
# ============================================================

observations_with_country_df = (
    observations_clean_df.alias("o")

    .join(
        country_lookup_df.alias("c"),

        F.col("o.entity_id_clean")
        == F.col("c.country_iso2_code"),

        "left"
    )
)

In [0]:
# ============================================================
# Validate country/entity dimension join
# ============================================================

# Observation rows that failed to resolve to a canonical
# Silver entity.
unmatched_country_observations_df = (
    observations_with_country_df

    .filter(
        F.col("canonical_entity_id").isNull()
    )
)

unmatched_country_observation_count = (
    unmatched_country_observations_df.count()
)


# Determine how many distinct source entities failed.
unmatched_country_entities_df = (
    unmatched_country_observations_df

    .select(

        F.col("entity_id_clean").alias(
            "source_entity_id"
        ),

        F.col("entity_iso3_code_clean").alias(
            "source_iso3_code"
        ),

        F.col("entity_name").alias(
            "source_entity_name"
        )
    )

    .distinct()

    .orderBy(
        "source_entity_name"
    )
)

unmatched_country_entity_count = (
    unmatched_country_entities_df.count()
)


print(
    f"Unmatched observation rows: "
    f"{unmatched_country_observation_count:,}"
)

print(
    f"Unmatched distinct entities: "
    f"{unmatched_country_entity_count:,}"
)


# Any unmatched source entity should be investigated.
if unmatched_country_entity_count > 0:

    display(
        unmatched_country_entities_df
    )

    raise RuntimeError(
        "Some observation entities failed to match "
        "the Silver country dimension."
    )

print(
    "Country dimension join validation passed."
)

In [0]:
# ============================================================
# Identify Silver entities that have no observation records
# ============================================================
#
# This is informational, not an error.
#
# The entity dimension contains 295 entities, while the
# selected World Bank indicators contain observations for
# only 265 entities.
# ============================================================

observed_source_entities_df = (
    observations_clean_df

    .select(
        F.col("entity_id_clean").alias(
            "observed_iso2_code"
        )
    )

    .distinct()
)


entities_without_observations_df = (
    countries_df.alias("c")

    .join(
        observed_source_entities_df.alias("o"),

        F.upper(
            F.trim(F.col("c.iso2_code"))
        )
        == F.col("o.observed_iso2_code"),

        "left_anti"
    )

    .select(
        "entity_id",
        "iso2_code",
        "entity_name",
        "entity_type",
        "region_name",
        "income_level_name"
    )

    .orderBy(
        "entity_type",
        "entity_name"
    )
)


entities_without_observations_count = (
    entities_without_observations_df.count()
)

print(
    f"Silver entities without observations: "
    f"{entities_without_observations_count:,}"
)

display(
    entities_without_observations_df
)

In [0]:
# ============================================================
# Enrich observations with governed indicator metadata
# ============================================================

observations_enriched_df = (
    observations_with_country_df.alias("o")

    .join(
        indicator_lookup_df.alias("i"),

        F.col("o.indicator_id_clean")
        == F.col("i.metadata_indicator_id"),

        "left"
    )
)

In [0]:
# ============================================================
# Validate indicator dimension join
# ============================================================

unmatched_indicator_df = (
    observations_enriched_df

    .filter(
        F.col("metadata_indicator_id").isNull()
    )
)

unmatched_indicator_count = (
    unmatched_indicator_df.count()
)

print(
    f"Observations with unmatched indicators: "
    f"{unmatched_indicator_count:,}"
)

if unmatched_indicator_count > 0:

    display(
        unmatched_indicator_df
        .select(
            "indicator_id_clean",
            "indicator_name"
        )
        .distinct()
    )

    raise RuntimeError(
        "Some observation indicators failed to match "
        "the Silver indicator dimension."
    )

print(
    "Indicator dimension join validation passed."
)

In [0]:
# ============================================================
# Create final Silver indicator observation fact table
# ============================================================
#
# Final grain:
#
#     entity_id + indicator_id + year
#
# IMPORTANT:
#
# entity_id is now the canonical identifier from our
# Silver country dimension.
#
# Example:
#
#     Source observation:
#         entity_id = IN
#
#     Silver fact:
#         entity_id = IND
#         source_entity_id = IN
#
# This gives downstream Gold tables and agents a consistent
# entity identifier while preserving source lineage.
# ============================================================

silver_observations_df = (
    observations_enriched_df

    .select(

        # ----------------------------------------------------
        # Canonical fact key
        # ----------------------------------------------------

        F.col("canonical_entity_id").alias(
            "entity_id"
        ),

        F.col("indicator_id_clean").alias(
            "indicator_id"
        ),

        "year",

        # ----------------------------------------------------
        # Economic observation
        # ----------------------------------------------------

        F.col("value").cast("double").alias(
            "value"
        ),

        # ----------------------------------------------------
        # Entity metadata
        # ----------------------------------------------------

        F.col("dimension_entity_name").alias(
            "entity_name"
        ),

        # Preserve original World Bank source identifier
        F.col("entity_id_clean").alias(
            "source_entity_id"
        ),

        # Preserve ISO3 supplied by observation endpoint
        # when available.
        F.col("entity_iso3_code_clean").alias(
            "source_iso3_code"
        ),

        # ISO2 identifier used for source-to-dimension join
        F.col("country_iso2_code").alias(
            "iso2_code"
        ),

        "entity_type",

        "region_id",
        "region_name",

        "admin_region_id",
        "admin_region_name",

        "income_level_id",
        "income_level_name",

        "lending_type_id",
        "lending_type_name",

        # ----------------------------------------------------
        # Indicator metadata
        # ----------------------------------------------------

        "indicator_display_name",
        "indicator_official_name",
        "indicator_category",

        # Preserve source observation indicator name
        F.col("indicator_name").alias(
            "source_indicator_name"
        ),

        # Metadata endpoint unit
        "metadata_unit",

        # Observation endpoint unit
        F.col("unit").alias(
            "source_unit"
        ),

        # ----------------------------------------------------
        # Observation metadata
        # ----------------------------------------------------

        "observation_status",
        "decimal",

        # ----------------------------------------------------
        # Source lineage
        # ----------------------------------------------------

        "source_system",
        "source_endpoint",

        "requested_start_year",
        "requested_end_year",

        "source_last_updated",
        "ingested_at",

        # Silver transformation timestamp
        "silver_processed_at"
    )
)

In [0]:
# ============================================================
# Validate Bronze -> Silver row preservation
# ============================================================
#
# Dimension enrichment must NOT:
#
#   - remove observations
#   - create additional observations
#
# Therefore Bronze and Silver row counts must be identical.
# ============================================================

silver_count = (
    silver_observations_df.count()
)

print(
    f"Bronze records: {bronze_count:,}"
)

print(
    f"Silver records: {silver_count:,}"
)

if silver_count != bronze_count:

    raise RuntimeError(
        "Bronze-to-Silver row count changed. "
        "A dimension join may have multiplied or removed rows."
    )

print(
    "Bronze-to-Silver row-count validation passed."
)

In [0]:
# ============================================================
# Validate final fact-table grain
# ============================================================
#
# Joining dimensions can accidentally multiply records.
#
# Therefore validate uniqueness again AFTER enrichment.
# ============================================================

final_duplicate_df = (
    silver_observations_df

    .groupBy(
        "entity_id",
        "indicator_id",
        "year"
    )

    .count()

    .filter(
        F.col("count") > 1
    )
)

final_duplicate_count = (
    final_duplicate_df.count()
)

print(
    f"Final duplicate fact keys: "
    f"{final_duplicate_count:,}"
)

if final_duplicate_count > 0:

    display(
        final_duplicate_df
    )

    raise RuntimeError(
        "Dimension enrichment created duplicate fact rows."
    )

print(
    "Final fact-grain validation passed."
)

In [0]:
# ============================================================
# Validate fact-table cardinality
# ============================================================

distinct_entities = (
    silver_observations_df
    .select("entity_id")
    .distinct()
    .count()
)

distinct_indicators = (
    silver_observations_df
    .select("indicator_id")
    .distinct()
    .count()
)

year_stats = (
    silver_observations_df
    .agg(
        F.min("year").alias("min_year"),
        F.max("year").alias("max_year")
    )
    .collect()[0]
)

min_year = year_stats["min_year"]
max_year = year_stats["max_year"]


print(
    f"Distinct entities:   {distinct_entities:,}"
)

print(
    f"Distinct indicators: {distinct_indicators:,}"
)

print(
    f"Minimum year:        {min_year}"
)

print(
    f"Maximum year:        {max_year}"
)


if distinct_entities != 265:
    raise RuntimeError(
        f"Expected 265 observed entities, "
        f"found {distinct_entities}."
    )

if distinct_indicators != 15:
    raise RuntimeError(
        f"Expected 15 indicators, "
        f"found {distinct_indicators}."
    )

if min_year < START_YEAR or max_year > END_YEAR:
    raise RuntimeError(
        "Unexpected year range detected."
    )

print(
    "Cardinality validation passed."
)

In [0]:
# ============================================================
# Validate observation coverage and NULL preservation
# ============================================================
#
# NULL and zero have different meanings:
#
#     NULL -> observation unavailable
#     0    -> actual reported value of zero
#
# We therefore preserve missing values exactly as supplied.
# ============================================================

value_stats = (
    silver_observations_df

    .agg(

        F.count("*").alias(
            "total_records"
        ),

        F.count("value").alias(
            "non_null_values"
        ),

        F.sum(
            F.when(
                F.col("value").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_values"
        )
    )

    .collect()[0]
)


total_records = value_stats["total_records"]
non_null_values = value_stats["non_null_values"]
null_values = value_stats["null_values"]


coverage_pct = (
    non_null_values
    / total_records
    * 100
    if total_records > 0
    else 0
)


print(
    f"Total records:   {total_records:,}"
)

print(
    f"Non-null values: {non_null_values:,}"
)

print(
    f"Null values:     {null_values:,}"
)

print(
    f"Coverage:        {coverage_pct:.2f}%"
)

In [0]:
# ============================================================
# Analyze data coverage by indicator
# ============================================================
#
# This helps identify indicators with sparse World Bank
# coverage and will later help the agent avoid assuming that
# every metric exists for every country/year.
# ============================================================

coverage_by_indicator_df = (
    silver_observations_df

    .groupBy(
        "indicator_id",
        "indicator_display_name",
        "indicator_category"
    )

    .agg(

        F.count("*").alias(
            "total_records"
        ),

        F.count("value").alias(
            "non_null_values"
        ),

        F.sum(
            F.when(
                F.col("value").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_values"
        )
    )

    .withColumn(
        "coverage_pct",
        F.round(
            F.col("non_null_values")
            / F.col("total_records")
            * 100,
            2
        )
    )

    .orderBy(
        F.desc("coverage_pct")
    )
)

display(
    coverage_by_indicator_df
)

In [0]:
# ============================================================
# Analyze data coverage by year
# ============================================================
#
# Recent years may have lower coverage because World Bank
# indicators have different reporting/publication lags.
# ============================================================

coverage_by_year_df = (
    silver_observations_df

    .groupBy(
        "year"
    )

    .agg(

        F.count("*").alias(
            "total_records"
        ),

        F.count("value").alias(
            "non_null_values"
        )
    )

    .withColumn(
        "null_values",
        F.col("total_records")
        - F.col("non_null_values")
    )

    .withColumn(
        "coverage_pct",
        F.round(
            F.col("non_null_values")
            / F.col("total_records")
            * 100,
            2
        )
    )

    .orderBy(
        "year"
    )
)

display(
    coverage_by_year_df
)

In [0]:
# ============================================================
# Analyze coverage by entity type
# ============================================================
#
# This separates:
#
#   COUNTRY_OR_ECONOMY
#   AGGREGATE
#
# and shows how much data is available for each.
# ============================================================

coverage_by_entity_type_df = (
    silver_observations_df

    .groupBy(
        "entity_type"
    )

    .agg(

        F.count("*").alias(
            "total_records"
        ),

        F.count("value").alias(
            "non_null_values"
        ),

        F.countDistinct(
            "entity_id"
        ).alias(
            "distinct_entities"
        )
    )

    .withColumn(
        "coverage_pct",
        F.round(
            F.col("non_null_values")
            / F.col("total_records")
            * 100,
            2
        )
    )

    .orderBy(
        "entity_type"
    )
)

display(
    coverage_by_entity_type_df
)

In [0]:
# ============================================================
# Inspect final Silver observation records
# ============================================================

display(
    silver_observations_df

    .select(
        "entity_id",
        "source_entity_id",
        "entity_name",
        "entity_type",
        "region_name",
        "indicator_id",
        "indicator_display_name",
        "indicator_category",
        "year",
        "value"
    )

    .orderBy(
        "entity_name",
        "indicator_id",
        "year"
    )

    .limit(100)
)

In [0]:
# ============================================================
# Sanity check: India GDP growth
# ============================================================
#
# Expected entity mapping:
#
#     canonical entity_id = IND
#     source_entity_id    = IN
#     source_iso3_code    = IND
#     iso2_code           = IN
#
# This proves that our source-to-canonical mapping works.
# ============================================================

display(
    silver_observations_df

    .filter(
        F.col("entity_id") == "IND"
    )

    .filter(
        F.col("indicator_id")
        == "NY.GDP.MKTP.KD.ZG"
    )

    .select(
        "entity_id",
        "source_entity_id",
        "source_iso3_code",
        "iso2_code",
        "entity_name",
        "indicator_display_name",
        "year",
        "value"
    )

    .orderBy(
        "year"
    )
)

In [0]:
# ============================================================
# Sanity check: Income aggregates
# ============================================================
#
# These source entities do not provide ISO3 codes in the
# observation endpoint:
#
#   XD -> High income
#   XM -> Low income
#   XN -> Lower middle income
#   XY -> Not classified
#   XT -> Upper middle income
#
# Our ISO2-based relationship should still resolve them.
# ============================================================

display(
    silver_observations_df

    .filter(
        F.col("source_entity_id").isin(
            "XD",
            "XM",
            "XN",
            "XY",
            "XT"
        )
    )

    .select(
        "entity_id",
        "source_entity_id",
        "source_iso3_code",
        "entity_name",
        "entity_type"
    )

    .distinct()

    .orderBy(
        "entity_name"
    )
)

In [0]:
# ============================================================
# FINAL PRE-WRITE VALIDATION
# ============================================================
#
# Do not write the Silver table unless every hard validation
# below passes.
# ============================================================

print("=" * 72)
print("SILVER INDICATOR OBSERVATION VALIDATION")
print("=" * 72)

print(
    f"Bronze records:                 "
    f"{bronze_count:,}"
)

print(
    f"Silver records:                 "
    f"{silver_count:,}"
)

print(
    f"Distinct entities:              "
    f"{distinct_entities:,}"
)

print(
    f"Distinct indicators:            "
    f"{distinct_indicators:,}"
)

print(
    f"Year range:                     "
    f"{min_year}-{max_year}"
)

print(
    f"Non-null values:                "
    f"{non_null_values:,}"
)

print(
    f"Null values:                    "
    f"{null_values:,}"
)

print(
    f"Overall coverage:               "
    f"{coverage_pct:.2f}%"
)

print(
    f"Duplicate fact keys:            "
    f"{final_duplicate_count:,}"
)

print(
    f"Unmatched indicators:           "
    f"{unmatched_indicator_count:,}"
)

print(
    f"Unmatched observation entities: "
    f"{unmatched_country_entity_count:,}"
)

print(
    f"Silver entities without obs:    "
    f"{entities_without_observations_count:,}"
)


# ------------------------------------------------------------
# Hard validation gates
# ------------------------------------------------------------

if silver_count != bronze_count:
    raise RuntimeError(
        "Bronze-to-Silver row count validation failed."
    )

if final_duplicate_count != 0:
    raise RuntimeError(
        "Duplicate fact keys detected."
    )

if unmatched_indicator_count != 0:
    raise RuntimeError(
        "Unmatched indicators detected."
    )

if unmatched_country_entity_count != 0:
    raise RuntimeError(
        "Unmatched observation entities detected."
    )

if distinct_entities != 265:
    raise RuntimeError(
        f"Expected 265 observed entities, "
        f"found {distinct_entities}."
    )

if distinct_indicators != 15:
    raise RuntimeError(
        f"Expected 15 indicators, "
        f"found {distinct_indicators}."
    )

if total_records != bronze_count:
    raise RuntimeError(
        "Observation count changed during transformation."
    )

print(
    "\nAll Silver indicator observation "
    "validations passed."
)

In [0]:
# ============================================================
# Write Silver indicator observations to Delta
# ============================================================

(
    silver_observations_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Saved Silver observations to:\n"
    f"{TARGET_TABLE}"
)

In [0]:
# ============================================================
# Validate persisted Delta table
# ============================================================
#
# We validate the table after persistence instead of assuming
# that a successful write means the result is correct.
# ============================================================

saved_df = spark.table(
    TARGET_TABLE
)

saved_count = (
    saved_df.count()
)


# Validate unique fact keys after persistence
saved_distinct_keys = (
    saved_df

    .select(
        "entity_id",
        "indicator_id",
        "year"
    )

    .distinct()

    .count()
)


# Validate non-null economic values
saved_non_null_values = (
    saved_df

    .filter(
        F.col("value").isNotNull()
    )

    .count()
)


# Validate missing economic values
saved_null_values = (
    saved_df

    .filter(
        F.col("value").isNull()
    )

    .count()
)


print(
    f"Expected records:      "
    f"{silver_count:,}"
)

print(
    f"Saved records:         "
    f"{saved_count:,}"
)

print(
    f"Distinct fact keys:    "
    f"{saved_distinct_keys:,}"
)

print(
    f"Saved non-null values: "
    f"{saved_non_null_values:,}"
)

print(
    f"Saved null values:     "
    f"{saved_null_values:,}"
)


# ------------------------------------------------------------
# Persisted-table validation gates
# ------------------------------------------------------------

if saved_count != silver_count:

    raise RuntimeError(
        "Silver write row-count validation failed."
    )


if saved_distinct_keys != saved_count:

    raise RuntimeError(
        "Fact-key uniqueness failed after write."
    )


if saved_non_null_values != non_null_values:

    raise RuntimeError(
        "Non-null value count changed after write."
    )


if saved_null_values != null_values:

    raise RuntimeError(
        "NULL value count changed after write."
    )


print(
    "Silver observation write validation passed."
)

In [0]:
# ============================================================
# Final notebook summary
# ============================================================

print("=" * 72)
print("WORLD BANK INDICATOR OBSERVATIONS SILVER TRANSFORMATION")
print("=" * 72)

print(
    f"Source:               "
    f"{OBSERVATIONS_SOURCE}"
)

print(
    f"Target:               "
    f"{TARGET_TABLE}"
)

print(
    f"Bronze records:       "
    f"{bronze_count:,}"
)

print(
    f"Silver records:       "
    f"{saved_count:,}"
)

print(
    f"Entities:             "
    f"{distinct_entities:,}"
)

print(
    f"Indicators:           "
    f"{distinct_indicators:,}"
)

print(
    f"Year range:           "
    f"{min_year}-{max_year}"
)

print(
    f"Non-null values:      "
    f"{saved_non_null_values:,}"
)

print(
    f"Null values:          "
    f"{saved_null_values:,}"
)

print(
    f"Coverage:             "
    f"{coverage_pct:.2f}%"
)

print(
    f"Duplicate fact keys:  "
    f"{final_duplicate_count:,}"
)

print(
    f"Unmatched entities:   "
    f"{unmatched_country_entity_count:,}"
)

print(
    f"Unmatched indicators: "
    f"{unmatched_indicator_count:,}"
)

print(
    "Layer:                Silver"
)

print(
    "Status:               SUCCESS"
)